# Testing New Ephemeris and SPICE loading

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import spiceypy as spice
import matplotlib.pyplot as plt

In [3]:
from contigo.solar_system_ephem import SPICEEphem
from contigo.solar_system_ephem import SolarSystemEnvironment
from contigo.forces.third_body_acc import ThirdBodyAcc
from contigo.forces.tba_utils import tba_pairwise_numba

### Test loading ephem with old version of loading ephem

In [ ]:
SPICEEphem()

In [ ]:
sw_e = pd.read_hdf("./data/ESA_pod.hdf")

In [ ]:
body = ['SUN','MOON']

j2000 = pd.Timestamp('2000-01-01 12:00:00')
spj2000 = ((sw_e['DateTime'] - j2000).dt.total_seconds()).to_list()

# set all needed attributes
et = np.array([spice.unitim(sp_in,'GPS','ET') for sp_in in spj2000])

unique_et, inv = np.unique(et, return_inverse=True)

# get the body positions in ecef
bd_ecef = np.array([spice.spkpos(bd,unique_et,'ITRF93','NONE','EARTH')[0]
            for bd in body])

bd_full = bd_ecef[:,inv,:]

In [ ]:
%%timeit
j2000 = pd.Timestamp('2000-01-01 12:00:00')
spj2000 = ((sw_e['DateTime'] - j2000).dt.total_seconds()).to_list()
#spj2000.extend(spj2000)

# set all needed attributes
et = np.array([spice.unitim(sp_in,'GPS','ET') for sp_in in spj2000])

In [ ]:
len(spj2000)

In [ ]:
et[0]

In [ ]:
print(spice.et2utc(et,'C',3))
print(sw_e['DateTime'][0])

In [ ]:
x = SPICEEphem()

In [ ]:
c_et, c_bd = x(body=['SUN','MOON'], et=et)

In [ ]:
np.array_equal(c_bd,bd_full)

### Test new Solar System Environement Class

SolarSystemEnvironment loads and caches ephemeris data with a time tolerance to save
on loading the same ephemeris data over and over.

In [ ]:
env = SolarSystemEnvironment(['SUN','MOON'], et=et[0:500], tolerance=0.1,provider=x)

In [ ]:
r_et, r_pos = env.get_ephem(et[0:500])
print(r_et.shape)
print(r_pos.shape)

In [ ]:
r_et, r_pos = env.get_ephem(et)
print(r_et.shape)
print(r_pos.shape)

#### Use ```get_ephem``` to add more ephem

In [ ]:
r_pos[id_max]

In [ ]:
r_et, r_pos = env.get_ephem(et)
print(r_et.shape)
print(r_pos.shape)
print(np.array_equal(r_pos, bd_full))

perc_err = 100*(r_pos.flatten()-bd_full.flatten())/bd_full.flatten()
id_max = np.unravel_index(perc_err.argmax(), r_pos.shape)

print(r_pos[id_max])
print(bd_full[id_max])

_ = plt.hist(perc_err,bins=50)

### Test Thirdbody Calculations and Look at error

In [ ]:
spos = sw_e[['x','y','z']].to_numpy()
stime = sw_e['DateTime']
x = sw_e['x'].to_numpy()
y = sw_e['y'].to_numpy()
z = sw_e['z'].to_numpy()
tba_cont = ThirdBodyAcc(spos=spos,stime=stime,body=['SUN','MOON'],scale='GPS')  
tba_cont.calc_tba()

tba_acc = tba_cont.get_tba()
print(tba_acc.shape)

In [ ]:
ephem_provider = SPICEEphem()
env = SolarSystemEnvironment(['SUN','MOON'], et=et, tolerance=None,provider=ephem_provider)
r_et, r_pos = env.get_ephem(et)
bd_acc = tba_pairwise_numba(spos, r_pos, tba_cont.GM)
print(bd_acc.shape)

In [ ]:
%%timeit
r_et, r_pos = env.get_ephem(et)

In [ ]:
print(bd_acc[1,0,:])
print(tba_acc[1,0,:])

In [ ]:
print(tba_acc[0,0,0])
print(bd_acc[0,0,0])

perc_err = 100*(bd_acc.flatten()-tba_acc.flatten())/tba_acc.flatten()
print(np.abs(perc_err).max())
_ = plt.hist(perc_err,bins=50)

In [ ]:
id_max = np.unravel_index(perc_err.argmax(), bd_acc.shape)

In [ ]:
print(bd_acc[id_max])
print(tba_acc[id_max])


In [ ]:
et = np.array([spice.unitim(sp_in,'GPS','TAI') for sp_in in spj2000])
et[0]

In [ ]:
spj2000[0]-et[0]

In [ ]:
# 2. Define your UTC time
utc_time = '2025-08-03T20:25:42'

# 3. Convert UTC to Ephemeris Time (ET/TDB)
et = spice.str2et(utc_time)

# 4. Convert ET to GPS time
gps_time = spice.unitim(et, 'ET', 'GPS')

print(f"UTC: {utc_time}")
print(f"GPS seconds past 1980-01-06 00:00:00: {gps_time}")

In [ ]:
spice.et2utc(et[0])

In [ ]:
et